# curl curl u + u = f 的有限元与 PINN 对比求解

## 问题描述

在 2D L-shape 域 $\Omega = [-1,1]^2 \setminus [0,1] \times [-1,0]$ 上求解:

$$\begin{cases}
\text{curl}\,\text{curl}\,\mathbf{u} + \mathbf{u} = \mathbf{f} & \text{in } \Omega \\
\mathbf{u} \cdot \mathbf{t} = 0 & \text{on } \partial\Omega
\end{cases}$$

**弱形式:**
$$(\text{curl}\,\mathbf{u}, \text{curl}\,\mathbf{v}) + (\mathbf{u}, \mathbf{v}) = (\mathbf{f}, \mathbf{v})$$

---
## Part 1: NGSolve 有限元求解 (Ground Truth)

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.geom2d import SplineGeometry
import numpy as np

# ===================== 1. 构造 L-shape 域 =====================
geo = SplineGeometry()
pnts = [
    (-1, -1),  # 0
    (0, -1),   # 1
    (0, 0),    # 2
    (1, 0),    # 3
    (1, 1),    # 4
    (-1, 1),   # 5
]
p = [geo.AppendPoint(*pt) for pt in pnts]

geo.Append(["line", p[0], p[1]], bc="bottom")
geo.Append(["line", p[1], p[2]], bc="reentrant_v")
geo.Append(["line", p[2], p[3]], bc="reentrant_h")
geo.Append(["line", p[3], p[4]], bc="right")
geo.Append(["line", p[4], p[5]], bc="top")
geo.Append(["line", p[5], p[0]], bc="left")

mesh = Mesh(geo.GenerateMesh(maxh=0.05))
print(f"Mesh elements: {mesh.ne}, vertices: {mesh.nv}")
Draw(mesh)


In [ ]:
# ===================== 2. HCurl 有限元空间与弱形式 =====================
order = 3
fes = HCurl(mesh, order=order, dirichlet=".*")
print(f"HCurl space, order={order}, ndof={fes.ndof}")

u, v = fes.TnT()

# 右端项 f
f_source = CoefficientFunction((
    sin(pi*y),
    sin(pi*x)
))

# 弱形式: (curl u, curl v) + (u, v) = (f, v)
a = BilinearForm(fes)
a += (curl(u) * curl(v) + u * v) * dx

f_form = LinearForm(fes)
f_form += f_source * v * dx

gfu = GridFunction(fes)
with TaskManager():
    a.Assemble()
    f_form.Assemble()
    gfu.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f_form.vec

print("FEM solve complete!")


In [ ]:
# ===================== 3. 可视化 FEM 解 =====================
Draw(gfu, mesh, "u_fem")
Draw(Norm(gfu), mesh, "u_norm_fem")


In [ ]:
# ===================== 4. 导出 VTK =====================
vtk = VTKOutput(mesh,
                coefs=[gfu, Norm(gfu)],
                names=["u_vec", "u_norm"],
                filename="curlcurl_fem_result",
                subdivision=2)
vtk.Do()
print("VTK exported: curlcurl_fem_result.vtu")


In [ ]:
# ===================== 5. 采样 FEM 解用于 PINN 训练 =====================

def sample_fem_solution(gfu, f_source, n_interior=8000, n_boundary=2000):
    mesh = gfu.space.mesh
    interior_pts, interior_u, interior_f = [], [], []
    count, attempts = 0, 0
    
    while count < n_interior and attempts < n_interior * 5:
        px = np.random.uniform(-1, 1)
        py = np.random.uniform(-1, 1)
        if px > 0 and py < 0:
            attempts += 1
            continue
        try:
            mp = mesh(px, py)
            val = gfu(mp)
            f_val = f_source(mp)
            interior_pts.append([px, py])
            interior_u.append([val[0], val[1]])
            interior_f.append([f_val[0], f_val[1]])
            count += 1
        except:
            pass
        attempts += 1
    
    boundary_pts, boundary_u = [], []
    edges = [
        ((-1,-1),(0,-1), n_boundary//6),
        ((0,-1),(0,0), n_boundary//6),
        ((0,0),(1,0), n_boundary//6),
        ((1,0),(1,1), n_boundary//6),
        ((1,1),(-1,1), n_boundary//6),
        ((-1,1),(-1,-1), n_boundary//6),
    ]
    for (x0,y0),(x1,y1),n in edges:
        for _ in range(n):
            t = np.random.uniform(0, 1)
            px, py = x0+t*(x1-x0), y0+t*(y1-y0)
            try:
                mp = mesh(px, py)
                val = gfu(mp)
                boundary_pts.append([px, py])
                boundary_u.append([val[0], val[1]])
            except:
                pass
    
    x_int = np.array(interior_pts)
    u_int = np.array(interior_u)
    f_int = np.array(interior_f)
    x_bnd = np.array(boundary_pts)
    u_bnd = np.array(boundary_u)
    print(f"Sampled {len(x_int)} interior, {len(x_bnd)} boundary points")
    return x_int, u_int, f_int, x_bnd, u_bnd

x_int, u_int, f_int, x_bnd, u_bnd = sample_fem_solution(gfu, f_source, 10000, 3000)
np.savez("fem_curlcurl_data.npz", x_int=x_int, u_int=u_int, f_int=f_int, x_bnd=x_bnd, u_bnd=u_bnd)
print("Saved fem_curlcurl_data.npz")


In [ ]:
# ===================== 6. 可视化采样点 =====================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sc0 = axes[0].scatter(x_int[:,0], x_int[:,1], c=np.sqrt(u_int[:,0]**2+u_int[:,1]**2), s=1, cmap='jet')
axes[0].set_title('Interior: |u_fem|'); axes[0].set_aspect('equal'); plt.colorbar(sc0, ax=axes[0])

sc1 = axes[1].scatter(x_int[:,0], x_int[:,1], c=u_int[:,0], s=1, cmap='jet')
axes[1].set_title('Interior: u_x (FEM)'); axes[1].set_aspect('equal'); plt.colorbar(sc1, ax=axes[1])

sc2 = axes[2].scatter(x_int[:,0], x_int[:,1], c=u_int[:,1], s=1, cmap='jet')
axes[2].set_title('Interior: u_y (FEM)'); axes[2].set_aspect('equal'); plt.colorbar(sc2, ax=axes[2])

plt.suptitle('FEM Solution Samples on L-shape Domain', fontsize=14)
plt.tight_layout(); plt.show()


---
## Part 2: PINN 求解 (以 FEM 解为 Ground Truth)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

data = np.load('fem_curlcurl_data.npz')
x_col = torch.tensor(data['x_int'], dtype=torch.float32).to(device)
u_fem = torch.tensor(data['u_int'], dtype=torch.float32).to(device)
f_rhs = torch.tensor(data['f_int'], dtype=torch.float32).to(device)
x_bc  = torch.tensor(data['x_bnd'], dtype=torch.float32).to(device)
u_bc_true = torch.tensor(data['u_bnd'], dtype=torch.float32).to(device)

print(f'Interior points: {x_col.shape}, Boundary points: {x_bc.shape}')


In [ ]:
# ===================== PINN 网络 =====================
class CubicReLU(nn.Module):
    def forward(self, x):
        return torch.pow(torch.relu(x), 3)

class VectorPINN(nn.Module):
    def __init__(self, activation_name='tanh', hidden_dim=128, n_layers=4):
        super().__init__()
        self.activation_name = activation_name
        if activation_name == 'tanh': act = nn.Tanh()
        elif activation_name == 'relu3': act = CubicReLU()
        elif activation_name == 'gelu': act = nn.GELU()
        else: act = nn.Tanh()
        
        layers = [nn.Linear(2, hidden_dim), act]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), act]
        layers.append(nn.Linear(hidden_dim, 2))
        self.net = nn.Sequential(*layers)
        
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        return self.net(x)


In [ ]:
# ===================== PDE 残差 =====================
def curlcurl_residual(model, x_in, f_rhs):
    x_in = x_in.detach().requires_grad_(True)
    u = model(x_in)
    u1, u2 = u[:, 0:1], u[:, 1:2]
    
    grad_u1 = torch.autograd.grad(u1, x_in, torch.ones_like(u1), create_graph=True)[0]
    grad_u2 = torch.autograd.grad(u2, x_in, torch.ones_like(u2), create_graph=True)[0]
    du1_dy, du2_dx = grad_u1[:, 1:2], grad_u2[:, 0:1]
    
    scalar_curl = du2_dx - du1_dy
    grad_curl = torch.autograd.grad(scalar_curl, x_in, torch.ones_like(scalar_curl), create_graph=True)[0]
    
    # curl curl u + u - f = 0
    res_x = grad_curl[:, 1:2] + u1 - f_rhs[:, 0:1]
    res_y = -grad_curl[:, 0:1] + u2 - f_rhs[:, 1:2]
    return res_x, res_y


In [ ]:
# ===================== 训练 Phase 1 =====================
ACT_TYPE = 'tanh'
model = VectorPINN(ACT_TYPE, hidden_dim=128, n_layers=4).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5000, gamma=0.5)

w_pde, w_bc, w_data = 1.0, 50.0, 1.0
loss_history = []

print(f'--- Training PINN ({ACT_TYPE}), params={sum(p.numel() for p in model.parameters())} ---')
for epoch in range(20001):
    optimizer.zero_grad()
    res_x, res_y = curlcurl_residual(model, x_col, f_rhs)
    loss_pde = torch.mean(res_x**2 + res_y**2)
    loss_bc = torch.mean((model(x_bc) - u_bc_true)**2)
    loss_data = torch.mean((model(x_col) - u_fem)**2)
    loss = w_pde*loss_pde + w_bc*loss_bc + w_data*loss_data
    loss.backward(); optimizer.step(); scheduler.step()
    loss_history.append(loss.item())
    if epoch % 2000 == 0:
        print(f'Epoch {epoch:5d}: Loss={loss.item():.4e} (PDE={loss_pde.item():.3e}, BC={loss_bc.item():.3e}, Data={loss_data.item():.3e})')


In [ ]:
# ===================== 训练 Phase 2 (fine-tune) =====================
optimizer2 = optim.Adam(model.parameters(), lr=1e-4)
scheduler2 = optim.lr_scheduler.StepLR(optimizer2, step_size=5000, gamma=0.5)

for epoch in range(20001, 40001):
    optimizer2.zero_grad()
    res_x, res_y = curlcurl_residual(model, x_col, f_rhs)
    loss_pde = torch.mean(res_x**2 + res_y**2)
    loss_bc = torch.mean((model(x_bc) - u_bc_true)**2)
    loss_data = torch.mean((model(x_col) - u_fem)**2)
    loss = w_pde*loss_pde + w_bc*loss_bc + w_data*loss_data
    loss.backward(); optimizer2.step(); scheduler2.step()
    loss_history.append(loss.item())
    if epoch % 2000 == 0:
        print(f'Epoch {epoch:5d}: Loss={loss.item():.4e} (PDE={loss_pde.item():.3e}, BC={loss_bc.item():.3e}, Data={loss_data.item():.3e})')


In [ ]:
# ===================== Loss 曲线 =====================
plt.figure(figsize=(10, 5))
plt.semilogy(loss_history)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title(f'Training Loss (curl curl u + u = f, {ACT_TYPE})')
plt.grid(True); plt.show()


In [ ]:
# ===================== 网格评估函数 =====================
def evaluate_on_grid(model, device, x_range, y_range, n_grid=120):
    xv = np.linspace(x_range[0], x_range[1], n_grid)
    yv = np.linspace(y_range[0], y_range[1], n_grid)
    X, Y = np.meshgrid(xv, yv)
    XY_torch = torch.tensor(np.stack([X.ravel(), Y.ravel()], 1), dtype=torch.float32).to(device)
    mask = (X > 0) & (Y < 0)
    with torch.no_grad():
        U = model(XY_torch).cpu().numpy()
    ux = U[:,0].reshape(n_grid, n_grid); uy = U[:,1].reshape(n_grid, n_grid)
    umag = np.sqrt(ux**2 + uy**2)
    ux[mask]=np.nan; uy[mask]=np.nan; umag[mask]=np.nan
    return X, Y, ux, uy, umag, mask

def evaluate_fem_on_grid(gfu, x_range, y_range, n_grid=120):
    msh = gfu.space.mesh
    xv = np.linspace(x_range[0], x_range[1], n_grid)
    yv = np.linspace(y_range[0], y_range[1], n_grid)
    X, Y = np.meshgrid(xv, yv)
    mask = (X > 0) & (Y < 0)
    ux_f = np.full_like(X, np.nan); uy_f = np.full_like(X, np.nan)
    for i in range(n_grid):
        for j in range(n_grid):
            if X[i,j]>0 and Y[i,j]<0: continue
            try:
                mp = msh(X[i,j], Y[i,j])
                val = gfu(mp)
                ux_f[i,j], uy_f[i,j] = val[0], val[1]
            except: pass
    umag_f = np.sqrt(np.nan_to_num(ux_f)**2 + np.nan_to_num(uy_f)**2)
    umag_f[mask] = np.nan
    return X, Y, ux_f, uy_f, umag_f

print('Evaluating on grid...')
ng = 100
Xp, Yp, uxp, uyp, up, mp = evaluate_on_grid(model, device, (-1,1), (-1,1), ng)
Xf, Yf, uxf, uyf, uf = evaluate_fem_on_grid(gfu, (-1,1), (-1,1), ng)
print('Done!')


In [ ]:
# ===================== FEM vs PINN 对比 =====================
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

im00 = axes[0,0].contourf(Xf, Yf, uf, levels=50, cmap='jet')
axes[0,0].set_title('FEM |u| (Ground Truth)'); axes[0,0].set_aspect('equal'); fig.colorbar(im00, ax=axes[0,0])

im01 = axes[0,1].contourf(Xp, Yp, up, levels=50, cmap='jet')
axes[0,1].set_title(f'PINN |u| ({ACT_TYPE})'); axes[0,1].set_aspect('equal'); fig.colorbar(im01, ax=axes[0,1])

err_mag = np.abs(up - uf)
im02 = axes[0,2].contourf(Xf, Yf, err_mag, levels=50, cmap='inferno')
axes[0,2].set_title('|Error| in |u|'); axes[0,2].set_aspect('equal'); fig.colorbar(im02, ax=axes[0,2])

im10 = axes[1,0].contourf(Xf, Yf, uxf, levels=50, cmap='jet')
axes[1,0].set_title(r'FEM $u_x$'); axes[1,0].set_aspect('equal'); fig.colorbar(im10, ax=axes[1,0])

im11 = axes[1,1].contourf(Xp, Yp, uxp, levels=50, cmap='jet')
axes[1,1].set_title(f'PINN $u_x$ ({ACT_TYPE})'); axes[1,1].set_aspect('equal'); fig.colorbar(im11, ax=axes[1,1])

err_ux = np.abs(uxp - uxf)
im12 = axes[1,2].contourf(Xf, Yf, err_ux, levels=50, cmap='inferno')
axes[1,2].set_title(r'|Error| in $u_x$'); axes[1,2].set_aspect('equal'); fig.colorbar(im12, ax=axes[1,2])

plt.suptitle(f'FEM vs PINN: curl curl u + u = f on L-shape, ACT={ACT_TYPE}, Epochs={len(loss_history)}', fontsize=14)
plt.tight_layout(); plt.show()

print(f'\n=== Error Statistics ===')
print(f'  Max |u| error:  {np.nanmax(err_mag):.6e}')
print(f'  Mean |u| error: {np.nanmean(err_mag):.6e}')
print(f'  L2 relative:    {np.sqrt(np.nansum(err_mag**2))/np.sqrt(np.nansum(uf**2)):.6e}')


In [ ]:
# ===================== Zoom-in 凹角附近 =====================
print('Evaluating zoom near re-entrant corner...')
nz = 100
Xpz, Ypz, _, _, upz, _ = evaluate_on_grid(model, device, (-0.2,0.2), (-0.2,0.2), nz)
Xfz, Yfz, _, _, ufz = evaluate_fem_on_grid(gfu, (-0.2,0.2), (-0.2,0.2), nz)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
im0 = axes[0].contourf(Xfz, Yfz, ufz, levels=50, cmap='jet')
axes[0].set_title('FEM |u| (Zoom 0.2)'); axes[0].set_aspect('equal'); plt.colorbar(im0, ax=axes[0])
im1 = axes[1].contourf(Xpz, Ypz, upz, levels=50, cmap='jet')
axes[1].set_title('PINN |u| (Zoom 0.2)'); axes[1].set_aspect('equal'); plt.colorbar(im1, ax=axes[1])
err_z = np.abs(upz - ufz)
im2 = axes[2].contourf(Xfz, Yfz, err_z, levels=50, cmap='inferno')
axes[2].set_title('|Error| (Zoom 0.2)'); axes[2].set_aspect('equal'); plt.colorbar(im2, ax=axes[2])
plt.suptitle('Zoom near re-entrant corner', fontsize=14); plt.tight_layout(); plt.show()
print(f'Zoom - Max err: {np.nanmax(err_z):.6e}, Mean err: {np.nanmean(err_z):.6e}')


In [ ]:
# ===================== u_y 分量对比 =====================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
im0 = axes[0].contourf(Xf, Yf, uyf, levels=50, cmap='jet')
axes[0].set_title(r'FEM $u_y$'); axes[0].set_aspect('equal'); plt.colorbar(im0, ax=axes[0])
im1 = axes[1].contourf(Xp, Yp, uyp, levels=50, cmap='jet')
axes[1].set_title(f'PINN $u_y$ ({ACT_TYPE})'); axes[1].set_aspect('equal'); plt.colorbar(im1, ax=axes[1])
err_uy = np.abs(uyp - uyf)
im2 = axes[2].contourf(Xf, Yf, err_uy, levels=50, cmap='inferno')
axes[2].set_title(r'|Error| in $u_y$'); axes[2].set_aspect('equal'); plt.colorbar(im2, ax=axes[2])
plt.suptitle(r'$u_y$ component comparison', fontsize=14); plt.tight_layout(); plt.show()


In [ ]:
# ===================== 保存模型 & 汇总 =====================
torch.save({'model_state_dict': model.state_dict(), 'activation': ACT_TYPE,
            'loss_history': loss_history, 'epochs': len(loss_history)}, 'pinn_curlcurl_model.pth')

print(f'\n{"="*50}')
print(f'Final Results Summary')
print(f'{"="*50}')
print(f'  Equation: curl curl u + u = f')
print(f'  Domain: L-shape')
print(f'  f = (sin(pi*y), sin(pi*x))')
print(f'  Activation: {ACT_TYPE}')
print(f'  Total epochs: {len(loss_history)}')
print(f'  Final loss: {loss_history[-1]:.6e}')
print(f'  Max |u| error:  {np.nanmax(err_mag):.6e}')
print(f'  Mean |u| error: {np.nanmean(err_mag):.6e}')
print(f'  L2 relative:    {np.sqrt(np.nansum(err_mag**2))/np.sqrt(np.nansum(uf**2)):.6e}')
